In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from collections import Counter
import random
import numpy as np
import csv
import os
import re
from nltk.tokenize import word_tokenize
import json
import torch.nn.functional as F
import pandas as pd

In [44]:
def evaluation_metrics(confusion_matrix):
  confusion_matrix = np.array(confusion_matrix)

  # print(confusion_matrix)

  accuracy = round(np.trace(confusion_matrix) / np.sum(confusion_matrix), 4)

  precision = np.diag(confusion_matrix) / np.sum(confusion_matrix, axis=0)
  recall = np.diag(confusion_matrix) / np.sum(confusion_matrix, axis=1)
  f1 = 2 * (precision * recall) / (precision + recall)

  micro_precision = np.sum(np.diag(confusion_matrix)) / np.sum(confusion_matrix)
  micro_recall = np.sum(np.diag(confusion_matrix)) / np.sum(confusion_matrix)
  micro_f1 = round(2 * (micro_precision * micro_recall) / (micro_precision + micro_recall), 4)

  macro_f1 = round(np.mean(f1), 4)
  return accuracy, precision, recall, f1, micro_f1, macro_f1

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, vocab_size, embedding_dim):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, input_seq, input_lengths):
        embedded = self.embedding(input_seq)
        embedded = embedded.float()  # Convert to float data type if needed
        packed_embedded = pack_padded_sequence(embedded, input_lengths, batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed_embedded)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        hidden = torch.cat((output[:, -1, :self.hidden_size], output[:, 0, self.hidden_size:]), dim=1)
        output = self.fc(hidden)
        return output


class CustomDataset(Dataset):
    def __init__(self, sentences, labels, word_to_index):
        self.sentences = sentences
        self.labels = labels
        self.word_to_index = word_to_index

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        # print(sentence)
        for word in sentence:
            if isinstance(word, list):
                print(word)
        indexed_sentence = [self.word_to_index[word] for word in sentence if word in self.word_to_index]
        label = self.labels[idx]
        return indexed_sentence, label

    def collate_fn(self, batch):
        sentences, labels = zip(*batch)
        lengths = [len(sentence) for sentence in sentences]
        max_length = max(lengths)
        padded_sentences = [sentence + [0] * (max_length - len(sentence)) for sentence in sentences]  # Padding
        return torch.LongTensor(padded_sentences), torch.LongTensor(labels)

df_tokenized = pd.read_pickle('task3df.pkl')
df_test = pd.read_pickle('task3df_test.pkl')


def get_data(df_tokenized):
    training_data = []

    for index, row in df_tokenized.iterrows():
        row1 = row['phrase1']
        row1.extend(row['phrase2'])
        training_data.append((row1, row['relation']))

    # Mapping words to indices
    word_to_index = {}
    for sent1, label in training_data:
        # print(sent1, sent2, label)
        # sent1.extend(sent2)

        for word in sent1:
            if word not in word_to_index:
                word_to_index[word] = len(word_to_index)
    word_to_index['PAD'] = len(word_to_index)
    word_to_index['UNK'] = len(word_to_index)


    train_data = [[], []]
    for sent1, label in training_data:
        train_data[0].append(sent1)
        train_data[1].append(label)

    index_to_word = {}
    for word in word_to_index:
        index_to_word[word_to_index[word]] = word

    return train_data, word_to_index, index_to_word


In [45]:
train_data, word_to_index, index_to_word = get_data(df_tokenized)
labels_train, sentences_train = train_data[1], train_data[0]
train_dataset = CustomDataset(sentences_train, labels_train, word_to_index)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=train_dataset.collate_fn)

# embedding_weights, word_to_index, index_to_word = get_embeddings()
input_size = len(word_to_index)
hidden_size = 256
output_size = 3
vocab_size = len(word_to_index)
embedding_dim = 100


model = LSTMClassifier(input_size, hidden_size, output_size, vocab_size, embedding_dim)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_dataloader, num_epochs, device):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs, [len(input_seq) for input_seq in inputs])
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_dataloader)
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

    return model

NUM_EPOCHS = 5
model = train(model, train_dataloader, NUM_EPOCHS, 'cuda')

Epoch 1/5, Loss: 0.1191
Epoch 2/5, Loss: 0.1017
Epoch 3/5, Loss: 0.0874
Epoch 4/5, Loss: 0.0778
Epoch 5/5, Loss: 0.0706


In [ ]:
validation_datalist, _, _ = get_data(df_test)
# print(validation_datalist[0][1])
# validation_datalist = test_data()
labels_val, sentences_val = validation_datalist[1], validation_datalist[0]

# print(type(word_to_index))
validation_dataset = CustomDataset(sentences_val, labels_val, word_to_index)
validation_dataloader = DataLoader(validation_dataset, batch_size=64, shuffle=False, collate_fn=validation_dataset.collate_fn)

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def calculate_performance_metrics(model, validation_dataloader, device):
    model.eval()
    true_labels = []
    predicted_labels = []

    with torch.no_grad():
        for inputs, labels in validation_dataloader:
            inputs = inputs.to(device)  # Move inputs to device
            # lengths = torch.LongTensor([len(seq) for seq in inputs]).to(device)  # Also move lengths to device

            outputs = model(inputs, [len(input_seq) for input_seq in inputs])
            _, predicted = torch.max(outputs, 1)

            true_labels.extend(labels.tolist())
            predicted_labels.extend(predicted.tolist())

    conf_matrix = confusion_matrix(true_labels, predicted_labels)
    accuracy, precision, recall, f1, micro_f1, macro_f1 = evaluation_metrics(conf_matrix)

    return accuracy, precision, recall, f1, micro_f1, macro_f1, conf_matrix

# Usage
accuracy, precision, recall, f1, micro_f1, macro_f1, conf_matrix = calculate_performance_metrics(model, validation_dataloader, 'cuda')


In [49]:
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:")
print(conf_matrix)

Precision: [0.47619048 0.56363636 0.99109559]
Recall: [0.31578947 0.27192982 0.99903203]
F1 Score: [0.37974684 0.36686391 0.99504799]
Confusion Matrix:
[[   30    17    48]
 [   29    31    54]
 [    4     7 11353]]
